<a href="https://colab.research.google.com/github/Data-Creater-Atlas/Data-Atlas/blob/jinho/Mission_2_0919.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

* lr = 3e - 4
* batch size = 32

* weights = ResNet18_Weights.IMAGENET1K_V1
* model = resnet18(weights=weights)   


# 환경 및 경로 설정

In [ ]:
!pip -q install ultralytics matplotlib opencv-python pandas

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 56.9 MB/s eta 0:00:00


In [ ]:
import os
from pathlib import Path
import json
import math
import cv2
import numpy as np
import pandas as pd
from PIL import Image

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

import torchvision
from torchvision import transforms
from torchvision.models import resnet18, ResNet18_Weights

## Google Drive Mounting

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## 절대 경로 설정

In [ ]:
# ---- 경로 정의 ----
DATA_ROOT = "/content/drive/MyDrive/Data_Creater_Camp"

# !! 아래 4개의 경로는 사용자 환경에 맞게 수정하세요 !!
Train_Source_DIR = f"{DATA_ROOT}/Training/01.원천데이터/TS_KS"       # 원본 학습 이미지 폴더
Train_Label_DIR  = f"{DATA_ROOT}/Training/02.라벨링데이터/TL_KS_LINE"  # 원본 학습 라벨(JSON) 폴더
Validation_Source_DIR = f"{DATA_ROOT}/Validation/01.원천데이터/VS_KS"  # 원본 검증 이미지 폴더
Validation_Label_DIR  = f"{DATA_ROOT}/Validation/02.라벨링데이터/VL_KS_LINE"  # 원본 검증 라벨(JSON) 폴더

## 데이터 전처리 데이터셋 디렉토리
* crop 파일 X, index.csv/labels만 생성

In [ ]:
DATASET_DIR = Path(DATA_ROOT) / "ResNet_Dataset"
for sub in ["train/images", "train/labels", "valid/images", "valid/labels"]:
    (DATASET_DIR / sub).mkdir(parents=True, exist_ok=True)

## 디바이스 설정

In [ ]:
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("DEVICE:", DEVICE)

DEVICE: cuda


# 데이터 전처리 및 관리 (JSON 파싱 + 인덱싱)

In [ ]:
def _safe_get(d: dict, key: str, default=None):
  # dict d에서 key가 없으면 값을 반환, 없거나 d가 dict가 아니면 default 반환
  return d[key] if (isinstance(d, dict) and key in d) else default

def load_line_labels(label_dir:str):
  # 라벨 JSON들이 저장된 디렉토리 경로를 받아서, 라인 정보 목록으로 파싱
  label_dir = Path(label_dir)
  json_files = sorted(label_dir.glob("*.json")) # 폴더 내의 .json 파일들을 정렬하여 수집
  items = []

  for jf in json_files:
    with open(jf, "r", encoding="utf-8") as f:
      data = json.load(f)

      if isinstance(data, dict):
        entries = list(data.values())
      elif isinstance(data, list):
        entries = data
      else:
        continue

      for entry in entries:  # 각 이미지 엔트리마다
        filename = _safe_get(entry, "filename", None) # 이미지 파일명
        regions = _safe_get(entry, "regions", [])     # 라벨링된 영역(region) 목록
        if not filename or not regions:               # 파일명 / 리전 없으면 스킵
          continue

        #각 region 마다 라인 혹은 polyline 처리
        for region in regions:
          shape = _safe_get(region, "shape_attributes", {})     # 도형 좌표 등
          attrs = _safe_get(region, "region_attributes", {})    # 사용자 정의 속정(높이, id 등)

          name = _safe_get(shape, "name", "").lower()           # 도형 타입(line, polyline 등)
          x1 = y1 = x2 = y2 = None                              # 좌표 초기화

          if name == "line":                                    # 도형 타입이 line이면
            x1 = float(_safe_get(shape, "x1", np.nan))          # 시작점 x1
            y1 = float(_safe_get(shape, "y1", np.nan))          # 시작점 y1
            x2 = float(_safe_get(shape, "x2", np.nan))          # 끝점 x2
            y2 = float(_safe_get(shape, "y2", np.nan))          # 끝점 y2

          elif name == "polyline":                              # 도형 타입이 polyline이면
            xs = _safe_get(shape, "all_point_x", [])            # x 좌표 배열
            ys = _safe_get(shape, "all_point_y", [])            # y 좌표 배열
            if isinstance(xs, list) and isinstance(ys, list) and len(xs) >= 2 and len(ys) >= 2:
              x1, y1 = float(xs[0]), float(ys[0])               # 폴리라인 첫 점
              x2, y2 = float(xs[-1]), float(ys[-1])             # 폴리라인 마지막 점

          else :
            continue                                            # line / polyline이 아니면 스킵

          if None in (x1, y1, x2, y2):                          # 좌표가 하나라도 비어 있으면 스킵
            continue

          # region_attributes에서 높이/ID 추출
          height = _safe_get(attrs, "height", None)             # 높이 값 추출
          chi_id = _safe_get(attrs, "chi_id", None)             # 굴뚝 ID (식별자) 추출

          # 숫자형 변환(가능하면)
          try:
            height = float(height)                              # 문자열 높이를 float으로 변환
          except:
          # 높이가 없는 경우 스킵
            continue

          items.append({
                    "img_path": filename,  # 절대경로 변환/상대경로 변환은 export 단계에서 처리
                    "x1": x1, "y1": y1, "x2": x2, "y2": y2,     # 라인 끝점 좌표
                    "height": height,                           # 타깃 (실수형 높이)
                    "chi_id": chi_id                            # 선택적 식별자
          })

  return items                                                  # 모든 JSON에서 수집한 라인 목록 반환

In [ ]:
def export_index_and_labels_only(items, image_dir: str, out_label_dir: str, out_index_csv: str, data_root: str):
  # items : JSON 파싱 결과(라인별 dict 리스트)
  # images_dir : 원본 이미지들이 있는 폴더 경로
  # out_label_dir : height 라벨, 텍스트 파일을 저장할 폴더
  # out_index_csv : index.csv 경로
  # data_root : 프로젝트 루트 (상대경로 기준점)



  image_dir = Path(image_dir)                       # 이미지 디렉토리를 Path 객체로 변환
  out_label_dir = Path(out_label_dir)               # 라벨 디렉터리를 Path 객체로 변환
  out_label_dir.mkdir(parents=True, exist_ok=True)  # 라벨 디렉터리 없으면 생성
  rows = []                                         # index.csv에 저장할 행 리스트


  # 이미지 파일명별 카운터 ( 같은 이미지에 여러 라인이 있을 경우 구분자 붙임)
  per_image_counter = {}

  for it in items:                                                      # 각 라인(region) 정보에 대체
    img_file = Path(image_dir) / it["img_path"]                         # 원래 JSON에 기록된 img_path를 기반으로 파일 경로 계산
    if not img_file.exists():                                           # 해당 경로에 파일이 없을 경우

      # 파일명이 경고 경로만 다른 경우를 대비해서 image_dir 아래에서 재탐색
      candidates = list(image_dir.rglob(Path(it["img_path"]).name))
      if candidates:
        img_file = candidates[0]                                        # 첫 번째 후보를 사용
      else:
        print(f"이미지를 찾을 수 없습니다 {it['img_path']}")
        continue

    stem = img_file.stem                                                # 확장자 없는 파일명
    # 같은 이미지에서 몇 번째 라인인지 카운터 증가
    per_image_counter[stem] = per_image_counter.get(stem, 0) + 1
    cnt = per_image_counter[stem]

    label_filename = f"{stem}_{cnt}.txt"                                # 라벨 파일명: <원본이름>_1.txt, _2.txt
    # 라벨 파일 쓰기 (height만)
    with open(out_label_dir / label_filename, "w", encoding="utf-8") as f:
      f.write(str(it["height"]))


    # DATA_ROOT 기준 상대 경로로 저장
    data_root_path = Path(data_root)
    try:
        rel_img_path = str(img_file.relative_to(data_root_path))
    except Exception:
        # 상대경로 계산 실패 시 절대경로 저장(최후수단)
        rel_img_path = str(img_file)

    # index.csv 한 줄(row) 구성
    rows.append({
        "img_path": rel_img_path,                                       # 원본 이미지 경로 (상대경로나 절대경로)
        "label_file": label_filename,                                   # height 라벨 텍스트 파일명
        "chi_id": it["chi_id"],                                         # 굴뚝 ID 등 식별자
        "height": it["height"],                                         # 높이 값
        "x1": it["x1"], "y1": it["y1"], "x2": it["x2"], "y2": it["y2"]  # 라인 좌표
    })


## 학습 / 검증 인데스 생성

In [ ]:
# pip install tqdm  (미설치 시)
from pathlib import Path
from tqdm.auto import tqdm
import csv

def export_index_and_labels_only_with_progress(
    items,
    source_dir: Path,
    labels_out_dir: Path,
    index_csv_path: Path,
    data_root: Path,
    desc: str = "Exporting",
):
    """
    items: 보통 (relative_path, label) 튜플의 리스트라고 가정.
           dict 형태일 수도 있어 방어적으로 처리.
    source_dir: 원본 이미지가 있는 최상위 디렉터리
    labels_out_dir: 라벨 .txt를 쓸 디렉터리 (없으면 생성)
    index_csv_path: index.csv를 쓸 경로
    data_root: 데이터셋 루트(상대경로 계산 등에 사용)
    desc: tqdm 진행바 설명 문자열
    """
    labels_out_dir.mkdir(parents=True, exist_ok=True)           # 라벨 저장 폴더
    index_csv_path.parent.mkdir(parents=True, exist_ok=True)    # index.csv 폴더 생성

    def _extract(item):
        # (path, label) 튜플 우선 가정
        if isinstance(item, (list, tuple)) and len(item) >= 2:
            return str(item[0]), float(item[1])
        # dict 방어
        if isinstance(item, dict):
            # 가능한 키 후보들
            # 여기서 'img_path'를 안 읽음 -> Load_line_Labels의 출력(dict)에선 이미지가 'img_path' 키라서 실패함
            p = item.get("path") or item.get("img") or item.get("image") or item.get("relpath")
            y = item.get("label") or item.get("height") or item.get("y") or item.get("value")
            return str(p), float(y)
        raise ValueError(f"지원하지 않는 items 원소 형식: {type(item)} → {item}")

    # index.csv: id, image_path, label_path, label_value 같은 열을 만든다고 가정
    rows = []
    with tqdm(total=len(items), desc=desc, unit="item") as pbar:
        for i, it in enumerate(items):
            rel_img_path, label_val = _extract(it)  # dict의 경우 'img_path'를 못 읽으면 rel_img_path가 될 수 있음

            # 라벨 파일 이름은 이미지 stem 기준으로 생성 (예: 000123.txt)
            # 같은 이미지에 라벨이 여러 개면 이 파일 하나에 계속 덮어씀 (_1, _2 없음)
            img_path = Path(rel_img_path)
            stem = img_path.stem
            label_txt_path = labels_out_dir / f"{stem}.txt"

            # 라벨 저장
            label_txt_path.write_text(f"{label_val}\n", encoding="utf-8")

            # index.csv 한 줄 (데이터 루트 기준의 상대경로로 저장하면 이식성이 좋아짐)
            img_abs = (source_dir / rel_img_path).resolve()
            img_rel_to_root = img_abs.relative_to(data_root.resolve())
            label_rel_to_root = label_txt_path.resolve().relative_to(data_root.resolve())

            rows.append({
                "id": i,
                "image_path": str(img_rel_to_root).replace("\\", "/"),
                "label_path": str(label_rel_to_root).replace("\\", "/"),
                "label_value": label_val,
            })

            pbar.update(1)

    # CSV 쓰기
    with index_csv_path.open("w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=["id", "image_path", "label_path", "label_value"])
        writer.writeheader()
        writer.writerows(rows)

    return {"count": len(rows), "index_csv": str(index_csv_path), "labels_dir": str(labels_out_dir)}

# ================== 사용 예시 (기존 코드 교체) ==================

train_items = load_line_labels(Train_Label_DIR)
valid_items = load_line_labels(Validation_Label_DIR)

train_index_csv = DATASET_DIR / "train" / "index.csv"
valid_index_csv = DATASET_DIR / "valid" / "index.csv"

train_result = export_index_and_labels_only_with_progress(
    train_items,
    Train_Source_DIR,
    DATASET_DIR / "train" / "labels",
    train_index_csv,
    DATA_ROOT,
    desc="Exporting (train)",
)

valid_result = export_index_and_labels_only_with_progress(
    valid_items,
    Validation_Source_DIR,
    DATASET_DIR / "valid" / "labels",
    valid_index_csv,
    DATA_ROOT,
    desc="Exporting (valid)",
)

print("[DONE] train:", train_result)
print("[DONE] valid:", valid_result)

KeyboardInterrupt: 

# PyTorch 데이터셋 / 데이터 로더

In [ ]:
def crop_line_patch_np(img_bgr: np.ndarray, x1, y1, x2, y2, out_size=224) -> np.ndarray:
    """
    입력: BGR 이미지, 라인 좌표
    동작:
      1) 라인 중심과 각도 계산
      2) 이미지 전체를 -angle 만큼 회전시켜 라인이 수평이 되도록 함
      3) 회전된 이미지에서 (라인 중심) 기준 224x224 패치를 crop
      4) 경계 이슈가 있으면 BORDER_REFLECT로 보정
      5) RGB로 변환 후 반환
    """
    H, W = img_bgr.shape[:2]
    # 중심 및 각도
    cx = (x1 + x2) * 0.5
    cy = (y1 + y2) * 0.5
    angle_rad = math.atan2((y2 - y1), (x2 - x1))
    angle_deg = np.degrees(angle_rad)

    # 회전 행렬 (반시계 기준이므로 수평 정렬을 위해 -angle)
    M = cv2.getRotationMatrix2D((cx, cy), -angle_deg, 1.0)
    rotated = cv2.warpAffine(img_bgr, M, (W, H), flags=cv2.INTER_LINEAR, borderMode=cv2.BORDER_REFLECT)

    # 회전 후 중심점 좌표
    center = np.array([cx, cy, 1.0], dtype=np.float32)
    rcx, rcy = M @ center

    # crop 범위 계산
    half = out_size // 2
    x_min = int(round(rcx - half))
    y_min = int(round(rcy - half))
    x_max = x_min + out_size
    y_max = y_min + out_size

    # 경계 보정: 필요 시 padding 후 crop
    pad_left = max(0, -x_min)
    pad_top = max(0, -y_min)
    pad_right = max(0, x_max - rotated.shape[1])
    pad_bottom = max(0, y_max - rotated.shape[0])

    if any([pad_left, pad_top, pad_right, pad_bottom]):
        rotated = cv2.copyMakeBorder(
            rotated, pad_top, pad_bottom, pad_left, pad_right, borderType=cv2.BORDER_REFLECT
        )
        x_min += pad_left
        x_max += pad_left
        y_min += pad_top
        y_max += pad_top

    patch = rotated[y_min:y_max, x_min:x_max]  # BGR
    # 혹시 패치 크기가 어긋났다면 리사이즈
    if patch.shape[0] != out_size or patch.shape[1] != out_size:
        patch = cv2.resize(patch, (out_size, out_size), interpolation=cv2.INTER_LINEAR)

    # BGR -> RGB
    patch_rgb = cv2.cvtColor(patch, cv2.COLOR_BGR2RGB)
    return patch_rgb

In [ ]:
class IndexCSVHeightDataset(Dataset):
    def __init__(self, index_csv: str, data_root: str, transform=None, forbid_aug=False):
        """
        index.csv를 읽어, on-the-fly로 라인 패치를 추출(검증셋: 증강금지).
        forbid_aug=True인 경우, transform에서 증강 요소가 있어도 실행 전 반드시 제거해야 함(여기서는 transform 구성 단계에서 보장).
        """
        self.df = pd.read_csv(index_csv)
        self.data_root = Path(data_root)
        self.transform = transform
        self.forbid_aug = forbid_aug

        # height 라벨 파일이 동일 폴더에 있지만, 값은 index.csv에 이미 존재(확인용) — 여기선 df['height']를 사용

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        # 원본 이미지 로딩
        img_path_rel = row["img_path"]
        img_path_abs = self.data_root / img_path_rel
        img_bgr = cv2.imread(str(img_path_abs), cv2.IMREAD_COLOR)
        if img_bgr is None:
            raise FileNotFoundError(f"Image not found: {img_path_abs}")

        # 좌표
        x1, y1, x2, y2 = row["x1"], row["y1"], row["x2"], row["y2"]
        # 패치 추출 (RGB)
        patch_rgb = crop_line_patch_np(img_bgr, x1, y1, x2, y2, out_size=224)

        # PIL 변환
        pil_img = Image.fromarray(patch_rgb)

        if self.transform is not None:
            pil_img = self.transform(pil_img)

        # 타겟 (height)
        height = float(row["height"])
        target = torch.tensor([height], dtype=torch.float32)

        return pil_img, target  # (C,H,W), (1,)

## Transform

In [ ]:
# ===========================
# Quick Fix: index.csv 복구 & 재생성  (with tqdm progress bars)
# ===========================
!pip -q install tqdm

from pathlib import Path
import pandas as pd
import json, os, glob
from tqdm import tqdm  # NEW

print("DATA_ROOT  :", DATA_ROOT)
print("TRAIN IMG  :", Train_Source_DIR)
print("TRAIN JSON :", Train_Label_DIR)
print("VALID IMG  :", Validation_Source_DIR)
print("VALID JSON :", Validation_Label_DIR)
print("DATASET_DIR:", DATASET_DIR)

# 1) 경로 존재 여부 점검
for p in [Train_Source_DIR, Train_Label_DIR, Validation_Source_DIR, Validation_Label_DIR]:
    print(f"[exists] {p} ->", os.path.exists(p))  # 폴더가 실제로 있는지 확인

# 2) 라벨 JSON 파일 수(재귀) 점검
train_jsons = sorted(Path(Train_Label_DIR).rglob("*.json"))  # 학습용 JSON 파일 전체 탐색
valid_jsons = sorted(Path(Validation_Label_DIR).rglob("*.json"))  # 검증용 JSON 파일 전체 탐색
print(f"#train json: {len(train_jsons)} | #valid json: {len(valid_jsons)}")  # 개수 출력

# 3) index 생성 함수 (tqdm 추가)
def _safe_get(d: dict, key: str, default=None):
    # dict에서 key가 있으면 반환, 없거나 dict이 아니면 default 반환
    return d[key] if (isinstance(d, dict) and key in d) else default

def load_line_labels_recursive(label_dir: str, desc="parse"):
    """
    기존 load_line_labels와 동일하지만 재귀 탐색 사용.
    tqdm 진행률 표시 추가.
    """
    items = []
    json_list = sorted(Path(label_dir).rglob("*.json"))   # 모든 JSON 파일 찾기
    for jf in tqdm(json_list, desc=f"[{desc}] JSON files", unit="file"):    # JSON 단위 진행률
        with open(jf, "r", encoding="utf-8") as f:
            data = json.load(f)      # JSON 읽기

        # VIA 형식(dict) -> values, 리스트 형식 -> 그대로
        entries = list(data.values()) if isinstance(data, dict) else (data if isinstance(data, list) else [])

        # region 단위 진행률 (entries 안에 regions가 있을 때만 대략적으로 표시)
        for entry in entries:
            filename = _safe_get(entry, "filename", None)
            regions = _safe_get(entry, "regions", [])
            if not filename or not regions:  # 파일명 / 리전 없으면 스킵
                continue

            for region in regions:   # region 단위
                shape = _safe_get(region, "shape_attributes", {})
                attrs = _safe_get(region, "region_attributes", {})
                name = str(_safe_get(shape, "name", "")).lower()  # 라인 / 폴리라인 구분


                if name == "line": # line 형식
                    x1 = float(_safe_get(shape, "x1", float("nan")))
                    y1 = float(_safe_get(shape, "y1", float("nan")))
                    x2 = float(_safe_get(shape, "x2", float("nan")))
                    y2 = float(_safe_get(shape, "y2", float("nan")))
                elif name == "polyline":   # polyline 형식
                    xs = _safe_get(shape, "all_points_x", [])
                    ys = _safe_get(shape, "all_points_y", [])
                    if not (isinstance(xs, list) and isinstance(ys, list) and len(xs) >= 2 and len(ys) >= 2):
                        continue
                    x1, y1, x2, y2 = float(xs[0]), float(ys[0]), float(xs[-1]), float(ys[-1])
                else:
                    continue    # 다른 도형은 스킵

                # 좌표 중 None/NaN 있으면 스킵
                if any(map(lambda v: v is None or (isinstance(v, float) and (pd.isna(v))), [x1,y1,x2,y2])):
                    continue

                # 높이 값 읽기
                height = _safe_get(attrs, "chi_height_m", None)
                try:
                    height = float(height)
                except:
                    continue      # 변환 실패 -> 스킵

                # items에 region 정보 저장
                items.append({
                    "img_path": filename,
                    "x1": x1, "y1": y1, "x2": x2, "y2": y2,
                    "height": height,
                    "chi_id": _safe_get(attrs, "chi_id", None)
                })
    return items

def export_index_and_labels_only(items, image_dir: str, out_label_dir: str, out_index_csv: str, data_root: str, desc="index"):
    """
    실제 크롭 이미지는 저장하지 않고, index.csv + height txt만 생성.
    tqdm 진행률 표시 추가.
    """
    from pathlib import Path
    import pandas as pd

    out_label_dir = Path(out_label_dir)
    out_label_dir.mkdir(parents=True, exist_ok=True)    # 라벨 폴더 생성
    rows, per_image_counter = [], {}                    # 결과 저장용ㅇ

    warn_missing = 0
    for it in tqdm(items, desc=f"[{desc}] build rows", unit="line"):
        img_file = Path(image_dir) / it["img_path"]
        if not img_file.exists():   # 원본 이미지 못 찾을 때
            # 파일명이 경로와 불일치시 이미지 디렉토리 전체를 재귀 검색
            cands = list(Path(image_dir).rglob(Path(it["img_path"]).name))
            if cands:
                img_file = cands[0]
            else:
                # 마지막으로, DATA_ROOT 전역에서도 찾아보기
                cands2 = list(Path(data_root).rglob(Path(it["img_path"]).name))
                if cands2:
                    img_file = cands2[0]
                else:
                    warn_missing += 1
                    if warn_missing <= 10:
                        print(f"[WARN] image not found: {it['img_path']}")
                    continue

        stem = img_file.stem
        # 같은 이미지에서 여러 라인 -> 카운터 붙여 파일명 구분
        per_image_counter[stem] = per_image_counter.get(stem, 0) + 1
        cnt = per_image_counter[stem]
        label_filename = f"{stem}_{cnt}.txt"

        # 라벨 텍스트 저장 (height만 저장)
        with open(out_label_dir / label_filename, "w", encoding="utf-8") as f:
            f.write(str(it["height"]))

        data_root_path = Path(data_root)
        try:
            rel_img_path = str(img_file.relative_to(data_root_path))
        except Exception:
            rel_img_path = str(img_file)

        # index.csv에 들어갈 row 구성
        rows.append({
            "img_path": rel_img_path,     # 이미지 경로
            "label_file": label_filename,  # 이미지 이름
            "chi_id": it["chi_id"],       # 굴뚝 아이디(식별자)
            "height": it["height"],       # 굴뚝 길이
            "x1": it["x1"], "y1": it["y1"], "x2": it["x2"], "y2": it["y2"],
        })

    if warn_missing > 10:
        print(f"[WARN] image not found (suppressed): {warn_missing - 10} more")

    # DataFrame 생성 -> CSV 저장
    df = pd.DataFrame(rows, columns=["img_path","label_file","chi_id","height","x1","y1","x2","y2"])
    Path(out_index_csv).parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(out_index_csv, index=False, encoding="utf-8")
    print(f"Saved index csv: {out_index_csv} ({len(df)} rows)")
    return df

# 4) 실제 재생성 수행 (진행률 표시)
train_items = load_line_labels_recursive(Train_Label_DIR, desc="train-parse")   # 학습 JSON 파싱
valid_items = load_line_labels_recursive(Validation_Label_DIR, desc="valid-parse")      # 검증 JSON 파싱
print(f"Parsed items -> train: {len(train_items)}, valid: {len(valid_items)}")

train_index_csv = DATASET_DIR / "train" / "index.csv"
valid_index_csv = DATASET_DIR / "valid" / "index.csv"

# 학습용 index.csv 재생성
_ = export_index_and_labels_only(
    train_items, Train_Source_DIR, DATASET_DIR / "train" / "labels", train_index_csv, DATA_ROOT, desc="train-index"
)
# 검증용 index.csv 재생성
_ = export_index_and_labels_only(
    valid_items, Validation_Source_DIR, DATASET_DIR / "valid" / "labels", valid_index_csv, DATA_ROOT, desc="valid-index"
)

# 5) 생성 결과 미리보기
if train_index_csv.exists():
    print("\n[train/index.csv head]")
    display(pd.read_csv(train_index_csv).head())
if valid_index_csv.exists():
    print("\n[valid/index.csv head]")
    display(pd.read_csv(valid_index_csv).head())

DATA_ROOT  : /content/drive/MyDrive/Data_Creater_Camp
TRAIN IMG  : /content/drive/MyDrive/Data_Creater_Camp/Training/01.원천데이터/TS_KS
TRAIN JSON : /content/drive/MyDrive/Data_Creater_Camp/Training/02.라벨링데이터/TL_KS_LINE
VALID IMG  : /content/drive/MyDrive/Data_Creater_Camp/Validation/01.원천데이터/VS_KS
VALID JSON : /content/drive/MyDrive/Data_Creater_Camp/Validation/02.라벨링데이터/VL_KS_LINE
DATASET_DIR: /content/drive/MyDrive/Data_Creater_Camp/ResNet_Dataset
[exists] /content/drive/MyDrive/Data_Creater_Camp/Training/01.원천데이터/TS_KS -> True
[exists] /content/drive/MyDrive/Data_Creater_Camp/Training/02.라벨링데이터/TL_KS_LINE -> True
[exists] /content/drive/MyDrive/Data_Creater_Camp/Validation/01.원천데이터/VS_KS -> True
[exists] /content/drive/MyDrive/Data_Creater_Camp/Validation/02.라벨링데이터/VL_KS_LINE -> True
#train json: 8052 | #valid json: 1006


[valid-parse] JSON files: 100%|██████████| 1006/1006 [00:26<00:00, 38.27file/s]


Parsed items -> train: 10590, valid: 1323


[train-index] build rows: 100%|██████████| 10590/10590 [2:05:03<00:00,  1.41line/s]


Saved index csv: /content/drive/MyDrive/Data_Creater_Camp/ResNet_Dataset/train/index.csv (10590 rows)


[valid-index] build rows: 100%|██████████| 1323/1323 [15:25<00:00,  1.43line/s]


Saved index csv: /content/drive/MyDrive/Data_Creater_Camp/ResNet_Dataset/valid/index.csv (1323 rows)

[train/index.csv head]


,img_path,label_file,chi_id,height,x1,y1,x2,y2
0,Training/01.원천데이터/TS_KS/K3A_CHN_20161112052404...,K3A_CHN_20161112052404_0_1.txt,1,76.78,108.0,378.0,184.0,370.0
1,Training/01.원천데이터/TS_KS/K3A_CHN_20161112052404...,K3A_CHN_20161112052404_0_2.txt,2,63.81,221.0,402.0,284.0,394.0
2,Training/01.원천데이터/TS_KS/K3A_CHN_20161112052404...,K3A_CHN_20161112052404_1_1.txt,1,76.78,109.0,122.0,185.0,114.0
3,Training/01.원천데이터/TS_KS/K3A_CHN_20161112052404...,K3A_CHN_20161112052404_1_2.txt,2,64.81,221.0,146.0,285.0,138.0
4,Training/01.원천데이터/TS_KS/K3A_CHN_20161112052404...,K3A_CHN_20161112052404_10_1.txt,1,105.08,100.0,236.0,204.0,225.0



[valid/index.csv head]


,img_path,label_file,chi_id,height,x1,y1,x2,y2
0,Validation/01.원천데이터/VS_KS/K3A_CHN_201611120524...,K3A_CHN_20161112052404_15_1.txt,1,137.59,132.0,412.0,268.0,396.0
1,Validation/01.원천데이터/VS_KS/K3A_CHN_201701150511...,K3A_CHN_20170115051130_1_1.txt,1,108.90,389.0,112.0,457.0,101.0
2,Validation/01.원천데이터/VS_KS/K3A_CHN_201701230521...,K3A_CHN_20170123052151_1_1.txt,1,107.28,328.0,137.0,382.0,128.0
3,Validation/01.원천데이터/VS_KS/K3A_CHN_201701230521...,K3A_CHN_20170123052151_14_1.txt,1,99.56,444.0,132.0,494.0,123.0
4,Validation/01.원천데이터/VS_KS/K3A_CHN_201701230521...,K3A_CHN_20170123052151_22_1.txt,1,113.09,140.0,173.0,197.0,164.0


In [ ]:
# ImageNet 정규화
IMAGENET_MEAN = [0.485, 0.456, 0.406]   # 채널별 평균값 (R, G, B)
IMAGENET_STD  = [0.229, 0.224, 0.225]   # 채널명 표준 편차

# 학습용: 검증셋은 별도로 augmentation 미적용 구성 사용
train_transform = transforms.Compose([
    # 회전, 색상, 반전 등 다양한 증강 기법 추가
    # 랜덤 회전 (15도 범위)
    transforms.RandomRotation(degrees=15),
    # 색상 변화: 밝기, 대비, 채도, 색상(Hue)을 랜덤하게 변형
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1),
    # 좌우 반전 확률 50%
    transforms.RandomHorizontalFlip(p=0.5),
    # 상하 반전 확률 50%
    transforms.RandomVerticalFlip(p=0.5),
    # 텐서 변환 (HWC -> CHW, [0, 255] -> [0, 1])
    transforms.ToTensor(),
    # ImageNet 통계값 정규화
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])


# 검증용: 증강 금지 (표준화만)
valid_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

# ---- Dataset & DataLoader ----
train_ds = IndexCSVHeightDataset(index_csv=str(train_index_csv),    # 학습 index.csv 파일 경로
                                 data_root=DATA_ROOT,   # 데이터셋 루트
                                 transform=train_transform,  # 위에서 정의한 학습용 변환
                                 forbid_aug=False   # 증강 O
                                 )

valid_ds = IndexCSVHeightDataset(index_csv=str(valid_index_csv),    # 검증 index.csv 파일 경로
                                 data_root=DATA_ROOT,
                                 transform=valid_transform,     # 검증용 변환 (정규화만)
                                 forbid_aug=True                # 증강 금지
                                 )

# 학습 DataLoader
train_loader = DataLoader(train_ds, batch_size=32, shuffle=True, # 배치 = 32, 셔플 사용
                          num_workers=2, pin_memory=True)
valid_loader = DataLoader(valid_ds, batch_size=32, shuffle=False, # 배치 사이즈 = 32, 셔플 X
                          num_workers=2, pin_memory=True)

len(train_ds), len(valid_ds)


(10590, 1323)

# 모델 학습(AMP) 및 평가 함수

In [ ]:
# 모델
weights = ResNet18_Weights.IMAGENET1K_V1
model = resnet18(weights=weights)   # ResNet18 : 모델 불러오기

# 마지막 FC 레이어(분류기)를 회귀용으로 교체
in_features = model.fc.in_features      # 원래 출력층 입력 차원
model.fc = nn.Linear(model.fc.in_features, 1)       # 출력: 스칼라 1개 (높이 예측)
model = model.to(DEVICE)

# Optim, Loss
criterion = nn.MSELoss()            # 손실 함수: 평균제곱오차(MSE)
optimizer = optim.Adam(model.parameters(), lr = 3e-4)   # Adam 옵티마이저 사용

MAX_EPOCHS = 80             # epoch 80회

# 스케줄러 추가 - CosineAnnealingLR
# T_max=MAX_EPOCHS - 한 사이클 길이
# eta_min=1e-6 - 최소 Leanring rate
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=MAX_EPOCHS, eta_min=1e-6)

# AMP - 학습 속도 / 메모리 절약
scaler = torch.cuda.amp.GradScaler(enabled=(DEVICE.type == "cuda"))

# Utilities
def rmse_from_mse(mse_val: float) -> float:   # MSE -> RMSE로 변환
  return float(np.sqrt(mse_val))

def train_one_epoch(model, loader, optimizer, device=DEVICE, epoch=None):
    model.train()             # 학습 모드 (Dropout/BN 활성화)
    total_loss, n = 0.0, 0    # 손실합, 샘플수 카운터

    #
    iterator = tqdm(loader,
                    desc=f"[train] epoch {epoch}" if epoch is not None else "[train]",
                    unit="batch", leave=False)

    for images, targets in iterator:
        images = images.to(device, non_blocking=True)     # 입력 이미지(GPU 이동)
        targets = targets.to(device, non_blocking=True)   # 정답 라벨(GPU 이동)

        optimizer.zero_grad(set_to_none=True)             # 그래디언트 초기화
        with torch.cuda.amp.autocast(enabled=(device.type == "cuda")):
            outputs = model(images)                       # 모델 forward
            loss = criterion(outputs, targets)            # MSE 손실 계산

        scaler.scale(loss).backward()                     # 역전파 (AMP scaling)
        scaler.step(optimizer)                            # optimizer 업데이트
        scaler.update()                                   # scale 갱신

        total_loss += loss.item() * images.size(0)        # 배치 손실 누적
        n += images.size(0)                               # 샘플 수 누적

        # 진행 중 RMSE를 간단히 표시(옵션)
        cur_mse = total_loss / max(1, n)
        iterator.set_postfix(rmse=np.sqrt(cur_mse))

    mse = total_loss / max(1, n)                          # 평균 MSE
    rmse = float(np.sqrt(mse))                            # RMSE 변환
    return rmse                                           # epoch RMSE 반환

@torch.no_grad()    # 검증 시 그래디언트 계산 비활성화
def evaluate(model, loader, device=DEVICE, desc="[valid]"):
    model.eval()                    # 평가 모드 (Dropout / BN 비활성화)
    preds_list, gts_list = [], []   # 예측 값/정답 값 저장 리스트
    total_loss, n = 0.0, 0

    for images, targets in tqdm(loader, desc=desc, unit="batch", leave=False):
        images = images.to(device, non_blocking=True)
        targets = targets.to(device, non_blocking=True)

        outputs = model(images)  # (B,1)
        loss = criterion(outputs, targets)
        total_loss += loss.item() * images.size(0)
        n += images.size(0)

        preds_list.append(outputs.detach().cpu().numpy())   # 예측 값 저장
        gts_list.append(targets.detach().cpu().numpy())      # 정답 값 저장

    mse = total_loss / max(1, n)      # 평균 MSE
    rmse = float(np.sqrt(mse))        # RMSE 변환
    preds = np.concatenate(preds_list, axis=0).reshape(-1)    # 예측 배열 병합
    gts = np.concatenate(gts_list, axis=0).reshape(-1)        # 정답 배열 병합
    return rmse, mse, preds, gts                      # 평가 지표/예측/정답 반환

class EarlyStopping:
    def __init__(self, patience=10, mode='min'):
        self.patience = patience              # 10 epoch 동안 개선 X -> stop
        self.mode = mode                      # 손실 최소
        self.best = None
        self.num_bad = 0                      # 개선 없는 epoch 수 카운트
        self.is_better = (lambda a, b: a < b) if mode == 'min' else (lambda a, b: a > b)

    def step(self, metric):
      # 첫 번째 호출이거나, metric이 개선된 경우
        if self.best is None or self.is_better(metric, self.best):
            self.best = metric      # best 갱신
            self.num_bad = 0        # bad 카운트 리셋
            return True             # improved
        else:
            self.num_bad += 1       # 개선 안됨 -> bad 카운트 증가
            return False  # not improved

Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 222MB/s]
/tmp/ipython-input-3218027474.py:20: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=(DEVICE.type == "cuda"))


## 모델 학습


In [ ]:
MAX_EPOCHS = 80
PATIENCE = 10
BEST_CKPT = str(DATASET_DIR / "resnet18_lineheight_best.pt")      # 모델 저장 경로

early = EarlyStopping(patience=PATIENCE, mode='min')              # 검증 RMSSE 최소화 기준
best_rmse = np.inf                                                # 현재까지 최상의 RMSE

for epoch in range(1, MAX_EPOCHS + 1):
    tr_rmse = train_one_epoch(model, train_loader, optimizer, DEVICE, epoch=epoch)
    va_rmse, va_mse, _, _ = evaluate(model, valid_loader, DEVICE, desc=f"[valid] epoch {epoch}")

    # 에포크 끝나면 스케줄러 업데이트
    scheduler.step()

    # 현재 epoch 결과 출력
    tqdm.write(f"[Epoch {epoch:03d}] train RMSE: {tr_rmse:.4f} | valid RMSE: {va_rmse:.4f}")

    if early.step(va_rmse):
        best_rmse = va_rmse
        torch.save(model.state_dict(), BEST_CKPT)
        tqdm.write(f"  ↳ New best! Saved checkpoint to: {BEST_CKPT}")
    else:
        if early.num_bad >= PATIENCE:
            tqdm.write(f"Early stopping triggered at epoch {epoch}. Best valid RMSE: {best_rmse:.4f}")
            break

[train] epoch 1:   0%|          | 0/331 [00:00<?, ?batch/s]/tmp/ipython-input-3218027474.py:38: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=(device.type == "cuda")):


[Epoch 001] train RMSE: 90.5758 | valid RMSE: 64.9026
  ↳ New best! Saved checkpoint to: /content/drive/MyDrive/Data_Creater_Camp/ResNet_Dataset/resnet18_lineheight_best.pt


[Epoch 002] train RMSE: 37.3529 | valid RMSE: 22.9216
  ↳ New best! Saved checkpoint to: /content/drive/MyDrive/Data_Creater_Camp/ResNet_Dataset/resnet18_lineheight_best.pt


[Epoch 003] train RMSE: 23.5099 | valid RMSE: 19.7026
  ↳ New best! Saved checkpoint to: /content/drive/MyDrive/Data_Creater_Camp/ResNet_Dataset/resnet18_lineheight_best.pt


[Epoch 004] train RMSE: 21.2398 | valid RMSE: 21.3339


...

[Epoch 076] train RMSE: 2.9937 | valid RMSE: 6.5914


[Epoch 077] train RMSE: 2.9247 | valid RMSE: 6.5347


[Epoch 078] train RMSE: 2.9468 | valid RMSE: 6.6269


[Epoch 079] train RMSE: 2.9513 | valid RMSE: 6.5779


[Epoch 080] train RMSE: 2.9591 | valid RMSE: 6.6025


# 성능 계산 (라인/이미지 라벨)

In [ ]:
model.load_state_dict(torch.load(BEST_CKPT, map_location=DEVICE))
model.to(DEVICE)

# 최종 검증 실행 (line 마다 단위 RMSE)
final_rmse, final_mse, preds, gts = evaluate(model, valid_loader, DEVICE)
print(f"\n[Final] Line-level RMSE: {final_rmse:.4f}")
# 굴뚝 하나씩 높이 예측 RMSE 출력

def rmse_image_level(preds: np.ndarray, gts: np.ndarray, index_csv_path: str):
    """
    검증 index.csv의 img_path를 기준으로 같은 이미지 내 여러 굴뚝(line)의
    예측/정답을 평균한 값으로 이미지 레벨 RMSE 계산.
    주의: DataLoader에서 shuffle=False여야 preds/gts 순서가 index.csv 순서와 일치.
    """
    df = pd.read_csv(index_csv_path)
    assert len(df) == len(preds) == len(gts), "index.csv 행 수와 다릅니다."

    df_eval = pd.DataFrame({
        "img_path": df["img_path"].values,
        "pred": preds,
        "gt": gts
    })

    grouped = df_eval.groupby("img_path").agg({"pred": "mean", "gt": "mean"}).reset_index()
    # 이미지 단위 MSE 계산
    mse = np.mean((grouped["pred"].values - grouped["gt"].values) ** 2)

    # RMSE 변환
    rmse = float(np.sqrt(mse))
    return rmse

이미지 라벨 RMSE 계산 및 출력
img_level_rmse = rmse_image_level(preds, gts, str(valid_index_csv))
print(f"[Final] Image-level RMSE: {img_level_rmse:.6f}")


[Final] Line-level RMSE: 6.4764
[Final] Image-level RMSE: 6.166430
